### __Import packages__

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
from scripts.PHMM.viterbi_algorithm.states import *
from scripts.PHMM.viterbi_algorithm.viterbi import *
from scripts.PHMM.distributions.AIS_RF_distribution import *
import pandas as pd

### __Get testing data__

In [3]:
test_set = pd.read_pickle("../data/processed/test_data/test_set.pkl")

### __Initialize states and model__

In [4]:
begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
model = PHMM(states, end)

### __Make test sequence (track 1039)__

In [5]:
sequence = test_set[test_set['TrackID'] == 1039]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None]

### __Run Viterbi model of first sequence__

In [6]:
path, probability = model.viterbi(AIS_sequence, RF_sequence)

In [7]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(30, 'RF'), (126, 'RF'), (195, 'RF'), (308, 'RF')],       
The indexes of RF and/or M of the real path are: [(30, 'RF'), (126, 'RF'), (195, 'RF'), (308, 'M')]


### __Make test sequence two (track 76)__

In [8]:
sequence2 = test_set[test_set['TrackID'] == 76]

AIS_sequence2 = [(time, coord) for time, coord in zip(sequence2["AIS_Timestamp"], sequence2["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence2 = [(time, coord) for time, coord in zip(sequence2["RF_Timestamp"], sequence2["RF"]) 
               if pd.notna(time) and coord is not None]

In [9]:
path2, probability2 = model.viterbi(AIS_sequence2, RF_sequence2)

In [10]:
RF_M_indexes_pred_path2 = [(i, state) for i, state in enumerate(path2) if state in {"RF", "M"}]
RF_M_indexes_real_path2 = [(i, state) for i, state in enumerate(sequence2["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path2}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path2}")

The indexes of RF and/or M of predicted path are: [(5, 'RF'), (150, 'RF')],       
The indexes of RF and/or M of the real path are: [(5, 'RF'), (150, 'RF')]


### __Make test sequence three (track 567)__

In [11]:
sequence3 = test_set[test_set['TrackID'] == 567]

AIS_sequence3 = [(time, coord) for time, coord in zip(sequence3["AIS_Timestamp"], sequence3["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence3 = [(time, coord) for time, coord in zip(sequence3["RF_Timestamp"], sequence3["RF"]) 
               if pd.notna(time) and coord is not None]

In [12]:
path3, probability3 = model.viterbi(AIS_sequence3, RF_sequence3)

In [13]:
RF_M_indexes_pred_path3 = [(i, state) for i, state in enumerate(path3) if state in {"RF", "M"}]
RF_M_indexes_real_path3 = [(i, state) for i, state in enumerate(sequence3["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path3}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path3}")

The indexes of RF and/or M of predicted path are: [(54, 'RF'), (394, 'RF'), (489, 'RF')],       
The indexes of RF and/or M of the real path are: [(54, 'RF'), (394, 'M'), (488, 'M')]


### __Run viterbi algorithm per RF observation__

In [14]:
from haversine import haversine, Unit
import math 

In [48]:
RF_data = test_set[test_set["RF"].notna()].copy()
RF_data["RF_data_point_id"] = RF_data.groupby("TrackID").cumcount() + 1

M_data = test_set[test_set["State"] == "M"].copy()
M_data["distance_km"] = M_data.apply(lambda row: haversine(row["AIS"], row["RF"], unit=Unit.KILOMETERS), axis=1)
radius_km = math.ceil(M_data["distance_km"].max())

viterbi_results = []

for _, RF_data_point in RF_data.iterrows():
    RF_coord = RF_data_point["RF"]
    RF_time = RF_data_point["RF_Timestamp"]
    
    for AIS_id, AIS_data in test_set.groupby("TrackID"):
        AIS_coords = [coord for coord in AIS_data["AIS"] if coord is not None]
        RF_within_radius = any(haversine(RF_coord, coord) <= radius_km for coord in AIS_coords)
        
        marge = pd.Timedelta(minutes=5)
        min_AIS_time = AIS_data["AIS_Timestamp"].min() - marge
        max_AIS_time = AIS_data["AIS_Timestamp"].max() + marge
        RF_within_time = min_AIS_time <= RF_time <= max_AIS_time
        
        if RF_within_radius and RF_within_time:
            AIS_seq = [(time, coord) 
                       for time, coord in zip(AIS_data["AIS_Timestamp"], AIS_data["AIS"]) 
                       if pd.notna(time) and coord is not None]
            RF_seq = [(RF_time, RF_coord)]
            
            path, prob = model.viterbi(AIS_seq, RF_seq)
            
            viterbi_results.append(
                {
                    "RF_sequence_id": RF_data_point["TrackID"],
                    "RF_data_point_id": RF_data_point["RF_data_point_id"],
                    "AIS_sequence_id": AIS_id,
                    "num_of_AIS_points": len(AIS_seq),
                    "viterbi_prob": prob,
                    "match": RF_data_point["TrackID"] == AIS_id,
                    "predicted_path": path
                })
            

In [90]:
def compute_penality(AIS_df, viterbi_score, weight):
    """
    Adjusts the Viterbi score based on the average log-speed of an AIS track
    In some AIS sequences, vessels may remain stationary for extended periods,
    resulting in minimal movement between positions but disproportionately high transition probabilities
    
    Args:
        sequence (pd.DataFrame): AIS data
        viterbi_score (float): The orginal log-probability score of the Viterbi algorithm
        weight (int): weight of the penalty, defaults to 1
    """
    log_speeds = []
    normal_speed = []
    time_coords = list(zip(AIS_df["AIS_Timestamp"], AIS_df["AIS"]))
    
    for i in range(1, len(time_coords)):
        prev_time, prev_coord = time_coords[i - 1]
        curr_time, curr_coord = time_coords[i]
        
        time_diff = (curr_time - prev_time).total_seconds()
        distance_diff = haversine(prev_coord, curr_coord, unit=Unit.KILOMETERS)
        speed = (distance_diff / time_diff) * 3600 if time_diff > 0 else 0.0
        
        log_speeds.append(np.log(speed + 1e-3))
        normal_speed.append(speed)
        
    mean_log_speed = np.mean(log_speeds) if log_speeds else 0
    mean_speed = np.mean(normal_speed) 
    adjusted_viterbi_score = viterbi_score + weight * mean_log_speed
    
    return mean_speed, adjusted_viterbi_score

In [91]:
for id, row in enumerate(viterbi_results):
    AIS_sequence_id = row["AIS_sequence_id"]
    
    AIS_sequence = test_set[
        (test_set["TrackID"] == AIS_sequence_id) &
        (test_set["AIS_Timestamp"].notna()) &
        (test_set["AIS"].notna())
    ]
    mean_speed, adjusted_viterbi_prob = compute_penality(AIS_sequence, row["viterbi_prob"], 3)
    viterbi_results[id]["adjusted_viterbi_prob"] = adjusted_viterbi_prob
    viterbi_results[id]["mean_speed"] = mean_speed
    viterbi_results[id]["normalized_viterbi_prob"] = adjusted_viterbi_prob/row["num_of_AIS_points"]

In [92]:
df_viterbi_results = pd.DataFrame(viterbi_results)
df_viterbi_results.head(5)

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,match,predicted_path,adjusted_viterbi_prob,normalized_viterbi_prob,mean_speed
0,23,1,23,669,-52.385595,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-46.629094,-0.069700,17.026579
1,23,2,23,669,-56.779936,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-51.023434,-0.076268,17.026579
2,23,3,23,669,-74.742474,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-68.985973,-0.103118,17.026579
3,29,1,29,308,-35.320747,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-41.508415,-0.134768,87.213526
4,29,1,1062,386,-33.600977,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-39.846815,-0.103230,1.432557


In [71]:
df_viterbi_results[df_viterbi_results["RF_sequence_id"] == 567]

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,match,predicted_path,adjusted_viterbi_prob,normalized_viterbi_prob,mean_speed
308,567,1,316,1118,-72.081512,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-75.664761,-0.067679,4.495181
309,567,1,567,908,-69.793220,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-62.777859,-0.069139,15.890228
310,567,2,316,1118,-76.028139,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-79.611389,-0.071209,4.495181
311,567,2,567,908,-90.034633,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-83.019272,-0.091431,15.890228
312,567,3,567,908,-78.195605,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-71.180245,-0.078392,15.890228


In [72]:
df_viterbi_results[df_viterbi_results["RF_sequence_id"] == 316]

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,match,predicted_path,adjusted_viterbi_prob,normalized_viterbi_prob,mean_speed
170,316,1,316,1118,-92.580753,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-96.164002,-0.086014,4.495181
171,316,1,567,908,-57.557100,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-50.541739,-0.055663,15.890228
172,316,2,316,1118,-107.696384,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-111.279633,-0.099535,4.495181
173,316,2,567,908,-69.427702,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-62.412342,-0.068736,15.890228
174,316,3,316,1118,-109.200609,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-112.783859,-0.100880,4.495181
175,316,3,567,908,-57.593065,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-50.577704,-0.055702,15.890228
176,316,4,316,1118,-109.765932,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-113.349182,-0.101386,4.495181
177,316,4,567,908,-53.636039,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-46.620678,-0.051344,15.890228


In [73]:
ids = df_viterbi_results.groupby(["RF_sequence_id", "RF_data_point_id"])["adjusted_viterbi_prob"].idxmax()
best_matches = df_viterbi_results.loc[ids].reset_index(drop=True)
best_matches

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,match,predicted_path,adjusted_viterbi_prob,normalized_viterbi_prob,mean_speed
0,23,1,23,669,-52.385595,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-46.629094,-0.069700,17.026579
1,23,2,23,669,-56.779936,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-51.023434,-0.076268,17.026579
2,23,3,23,669,-74.742474,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-68.985973,-0.103118,17.026579
3,29,1,1062,386,-33.600977,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-39.846815,-0.103230,1.432557
4,29,2,29,308,-30.637422,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-36.825090,-0.119562,87.213526
...,...,...,...,...,...,...,...,...,...,...
695,1351,1,1351,463,-41.324335,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-44.468026,-0.096043,4.518623
696,1351,2,1351,463,-44.822784,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-47.966474,-0.103599,4.518623
697,1354,1,1354,605,-53.898922,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-56.706064,-0.093729,7.631973
698,1354,2,1354,605,-54.182703,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-56.989845,-0.094198,7.631973


In [85]:
best_matches["mean_speed"].idxmax()

344

In [86]:
best_matches.iloc[344]

RF_sequence_id                                                           673
RF_data_point_id                                                           1
AIS_sequence_id                                                          673
num_of_AIS_points                                                        876
viterbi_prob                                                      -84.014034
match                                                                   True
predicted_path             [begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...
adjusted_viterbi_prob                                             -74.372492
normalized_viterbi_prob                                              -0.0849
mean_speed                                                       1852.956341
Name: 344, dtype: object

In [83]:
best_matches["mean_speed"].min()

np.float64(0.005084386063617209)

In [75]:
len(best_matches[best_matches["match"] == False])

48

In [41]:
ids = df_viterbi_results.groupby(["RF_sequence_id", "RF_data_point_id"])["viterbi_prob"].idxmax()
best_matches2 = df_viterbi_results.loc[ids].reset_index(drop=True)
best_matches2

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,viterbi_prob,match,predicted_path,adjusted_viterbi_prob
0,23,1,23,-52.385595,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-48.547928
1,23,2,23,-56.779936,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-52.942268
2,23,3,23,-74.742474,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-70.904807
3,29,1,1062,-33.600977,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-37.764869
4,29,2,29,-30.637422,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-34.762534
...,...,...,...,...,...,...,...
695,1351,1,1351,-41.324335,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-43.420129
696,1351,2,1351,-44.822784,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-46.918577
697,1354,1,1354,-53.898922,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-55.770350
698,1354,2,1354,-54.182703,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",-56.054131


In [25]:
test_set[test_set["TrackID"] == 316].iloc[85:90]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
34107,316,316009457,2,2024-01-03 01:40:11,2024-01-03 01:40:11,NaT,"(42.97816, -82.41604)",None,AIS
34108,316,316009457,2,2024-01-03 01:41:21,2024-01-03 01:41:21,2024-01-03 01:41:00,"(42.97494, -82.41562)","(42.97441698379122, -82.42634931899319)",M
34109,316,316009457,2,2024-01-03 01:42:30,2024-01-03 01:42:30,NaT,"(42.97194, -82.41643)",None,AIS
34110,316,316009457,2,2024-01-03 01:43:40,2024-01-03 01:43:40,NaT,"(42.96888, -82.4174)",None,AIS
34111,316,316009457,2,2024-01-03 01:44:50,2024-01-03 01:44:50,NaT,"(42.96599, -82.41887)",None,AIS


In [26]:
test_set[test_set["TrackID"] == 567].iloc[50:55]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
60005,567,366904890,2,2024-01-03 01:30:03,2024-01-03 01:30:03,NaT,"(42.3032, -83.08831)",None,AIS
60006,567,366904890,2,2024-01-03 01:33:04,2024-01-03 01:33:04,NaT,"(42.30322, -83.08829)",None,AIS
60007,567,366904890,2,2024-01-03 01:36:04,2024-01-03 01:36:04,NaT,"(42.3032, -83.08832)",None,AIS
60008,567,366904890,2,2024-01-03 01:39:03,2024-01-03 01:39:03,NaT,"(42.30321, -83.08833)",None,AIS
60009,567,366904890,2,2024-01-03 01:41:00,NaT,2024-01-03 01:41:00,None,"(42.3115251103819, -83.08503303997854)",RF


In [44]:
time = (pd.Timestamp("2024-01-03 01:41:00") - pd.Timestamp("2024-01-03 01:40:11")).total_seconds()
dist = haversine((42.97441698379122, -82.42634931899319), (42.97816, -82.41604))
AIS_RF_probability(time, dist)

np.float64(0.9740481961609083)

In [45]:
time = (pd.Timestamp("2024-01-03 01:41:00") - pd.Timestamp("2024-01-03 01:39:03")).total_seconds()
dist = haversine((42.97441698379122, -82.42634931899319), (42.30321, -83.08833))
AIS_RF_probability(time, dist)

np.float64(0.24356335140959184)

In [109]:
test_set[(test_set["TrackID"] == 673) & (test_set["AIS_Timestamp"] == "2024-01-25 20:48:09")].iloc[76092-2: 76092+ 2]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State


In [110]:
test_set.iloc[76092-2: 76092+ 2]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
76090,673,373485000,0,2024-01-25 20:46:42,2024-01-25 20:46:42,NaT,"(34.43152, -121.52372)",None,AIS
76091,673,373485000,0,2024-01-25 20:47:43,2024-01-25 20:47:43,NaT,"(34.43193, -121.51973)",None,AIS
76092,673,373485000,0,2024-01-25 20:48:09,2024-01-25 20:48:09,NaT,"(33.96917, -125.32927)",None,AIS
76093,673,373485000,0,2024-01-25 20:48:52,2024-01-25 20:48:52,NaT,"(34.43242, -121.51517)",None,AIS


In [97]:
AIS_673 = test_set[(test_set["TrackID"] == 673) & (test_set["AIS_Timestamp"].notna()) &(test_set["AIS"].notna())]
time_coords = list(zip(AIS_673["AIS_Timestamp"], AIS_673["AIS"]))
time_coords[:5], len(time_coords)

([(Timestamp('2024-01-24 22:10:39'), (33.87292, -127.39537)),
  (Timestamp('2024-01-24 22:12:03'), (33.87312, -127.38792)),
  (Timestamp('2024-01-24 22:13:57'), (33.87337, -127.37782)),
  (Timestamp('2024-01-24 22:23:39'), (33.87463, -127.32618)),
  (Timestamp('2024-01-24 22:24:58'), (33.8748, -127.31917))],
 876)

In [100]:
time_coords[-1:]

[(Timestamp('2024-01-25 22:10:32'), (34.46585, -121.17982))]

In [101]:
for i in range(1, len(time_coords)):
    prev_time, prev_coord = time_coords[i - 1]
    curr_time, curr_coord = time_coords[i]
    print(f"Previous time: {prev_time}, Previous coord: {prev_coord}")
    print(f"Current time: {curr_time}, Current coord: {curr_coord}")    
    
    time_diff = (curr_time - prev_time).total_seconds()
    distance_diff = haversine(curr_coord, prev_coord, unit=Unit.KILOMETERS)
    print(f"Time difference: {time_diff} seconds, Distance difference: {distance_diff} km")
    # speed = (distance_diff / time_diff) * 3600 if time_diff > 0 else 0.0
    
    # print(f"Time difference: {time_diff} seconds, Distance difference: {distance_diff} km, Speed: {speed} km/h")

Previous time: 2024-01-24 22:10:39, Previous coord: (33.87292, -127.39537)
Current time: 2024-01-24 22:12:03, Current coord: (33.87312, -127.38792)
Time difference: 84.0 seconds, Distance difference: 0.6881618861785299 km
Previous time: 2024-01-24 22:12:03, Previous coord: (33.87312, -127.38792)
Current time: 2024-01-24 22:13:57, Current coord: (33.87337, -127.37782)
Time difference: 114.0 seconds, Distance difference: 0.9328688349654652 km
Previous time: 2024-01-24 22:13:57, Previous coord: (33.87337, -127.37782)
Current time: 2024-01-24 22:23:39, Current coord: (33.87463, -127.32618)
Time difference: 582.0 seconds, Distance difference: 4.769536142314554 km
Previous time: 2024-01-24 22:23:39, Previous coord: (33.87463, -127.32618)
Current time: 2024-01-24 22:24:58, Current coord: (33.8748, -127.31917)
Time difference: 79.0 seconds, Distance difference: 0.6474437190567326 km
Previous time: 2024-01-24 22:24:58, Previous coord: (33.8748, -127.31917)
Current time: 2024-01-24 22:28:46, Cur

In [98]:
AIS_673

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
75297,673,373485000,0,2024-01-24 22:10:39,2024-01-24 22:10:39,NaT,"(33.87292, -127.39537)",None,AIS
75298,673,373485000,0,2024-01-24 22:12:03,2024-01-24 22:12:03,NaT,"(33.87312, -127.38792)",None,AIS
75299,673,373485000,0,2024-01-24 22:13:57,2024-01-24 22:13:57,NaT,"(33.87337, -127.37782)",None,AIS
75300,673,373485000,0,2024-01-24 22:23:39,2024-01-24 22:23:39,NaT,"(33.87463, -127.32618)",None,AIS
75301,673,373485000,0,2024-01-24 22:24:58,2024-01-24 22:24:58,NaT,"(33.8748, -127.31917)",None,AIS
...,...,...,...,...,...,...,...,...,...
76168,673,373485000,0,2024-01-25 22:06:01,2024-01-25 22:06:01,NaT,"(34.4652, -121.20328)",None,AIS
76169,673,373485000,0,2024-01-25 22:07:12,2024-01-25 22:07:12,NaT,"(34.46543, -121.19402)",None,AIS
76170,673,373485000,0,2024-01-25 22:08:13,2024-01-25 22:08:13,NaT,"(34.46557, -121.18977)",None,AIS
76171,673,373485000,0,2024-01-25 22:09:21,2024-01-25 22:09:21,NaT,"(34.46568, -121.18482)",None,AIS


In [ ]:
time_coords = list(zip(AIS_df["AIS_Timestamp"], AIS_df["AIS"]))
    
    for i in range(1, len(time_coords)):
        prev_time, prev_coord = time_coords[i - 1]
        curr_time, curr_coord = time_coords[i]
        
        time_diff = (curr_time - prev_time).total_seconds()
        distance_diff = haversine(prev_coord, curr_coord, unit=Unit.KILOMETERS)
        speed = (distance_diff / time_diff) * 3600 if time_diff > 0 else 0.0
        
        log_speeds.append(np.log(speed + 1e-3))
        normal_speed.append(speed)